In [ ]:
/**
 * ============================================================================
 * DISTRIBUTED TRANSACTIONS — EXPLAINED LIKE WE'RE TALKING
 * ============================================================================
 *
 * Picture someone clicking "Place Order" on a shopping site. Behind the
 * scenes, THREE things need to happen together:
 *   1. Inventory Service reserves the items
 *   2. Payment Service charges the customer's card
 *   3. Order Service creates a record of the purchase
 *
 * All three need to succeed together, or NONE of them should happen.
 *
 * Now imagine the payment goes through, but updating inventory fails.
 * Oops — you just charged someone for something you can't actually send
 * them. Or the order record gets created, but the payment fails — now
 * you've promised goods without getting paid.
 *
 * If everything lived in ONE database, this is easy: wrap it all in a
 * single transaction, and the database gives you "all or nothing" for
 * free. But once the data is spread across DIFFERENT services (each with
 * its own database), you no longer have that safety net built in. You
 * have to build it yourself. That's what this whole topic — "distributed
 * transactions" — is about.
 *
 *
 * ============================================================================
 * 1. WHAT IS A DISTRIBUTED TRANSACTION?
 * ============================================================================
 *
 * Simple definition: it's a transaction-like process that spans MULTIPLE
 * services, databases, or systems — where the goal is still "all changes
 * happen, or none do" (or, if something goes wrong partway, there's a
 * reliable way to undo/compensate for the partial work).
 *
 * WHY THIS IS DIFFERENT FROM A NORMAL (LOCAL) TRANSACTION:
 *
 *   LOCAL TRANSACTION (all in ONE database):
 *
 *       BEGIN
 *         UPDATE inventory
 *         INSERT order
 *         UPDATE payment
 *       COMMIT
 *       (the database itself guarantees all-or-nothing — automatic!)
 *
 *   DISTRIBUTED TRANSACTION (spread across services):
 *
 *       [Transaction Coordinator]
 *              |         |         |
 *       [Service A DB] [Service B DB] [Service C DB]
 *       (nobody automatically guarantees "all or nothing" here —
 *        YOU have to build that coordination yourself)
 *
 * In a LOCAL transaction, the database gives you these 4 guarantees for
 * free (often remembered by the acronym ACID):
 *   - Atomicity:   everything succeeds together, or nothing does
 *   - Consistency: the database always moves from one valid state to another
 *   - Isolation:   transactions happening at the same time don't mess
 *                  with each other
 *   - Durability:  once something is "committed," it survives crashes
 *
 * Once your data is SPLIT across multiple services, no single database
 * can guarantee these anymore. Each service only controls ITS OWN piece.
 * The really hard part: several independent systems now need to agree
 * on "did we commit, or did we abort?" — while ANY of them (or the
 * network between them) could fail at literally the worst possible moment.
 *
 *
 * ============================================================================
 * 2. WHY THIS IS SO HARD (the core obstacles)
 * ============================================================================
 *
 * -----------------------------------------------------------------------
 * 2.1 THE "TWO GENERALS" PROBLEM (a classic thought experiment)
 * -----------------------------------------------------------------------
 * Imagine two army generals camped on opposite sides of a valley. They
 * need to attack the city between them AT THE SAME TIME, or they'll both
 * lose. But they can only talk by sending messengers through the valley
 * — and messengers might get captured along the way.
 *
 *   General A sends: "Attack at dawn!"
 *   General B thinks: "Got it — but does A know I got it?" So B replies:
 *   General B sends: "Acknowledged, I'll attack!"
 *   General A thinks: "Great — but does B know I got THEIR reply?"
 *   ...and so on, forever.
 *
 * PICTURE:
 *
 *   [General A] --"Attack at dawn"--> [General B]
 *   [General A] <--"Acknowledged!"---- [General B]
 *   Did A receive that? Did B receive A's next confirmation? Neither
 *   general can EVER be 100% sure the other got the very last message.
 *
 * Sending more messages back and forth reduces the doubt, but it NEVER
 * fully removes it. This is exactly the same problem two computer
 * services face when trying to agree "should we commit, or not?" — any
 * message between them might get lost, and either one might crash.
 *
 * -----------------------------------------------------------------------
 * 2.2 NETWORKS ARE UNRELIABLE (in sneaky ways)
 * -----------------------------------------------------------------------
 * Networks can:
 *   - lose messages completely (never arrive at all)
 *   - delay messages (show up way later than expected)
 *   - duplicate messages (the same message arrives twice)
 *   - reorder messages (arrive in the wrong order)
 *
 * The really annoying part: if you send a message and get NO response,
 * you genuinely cannot tell WHY. Maybe your message got lost. Maybe
 * their reply got lost. Maybe they're just slow. Maybe they crashed.
 * From where you're standing, all of these LOOK identical.
 *
 * -----------------------------------------------------------------------
 * 2.3 SERVERS CAN CRASH AT THE WORST POSSIBLE MOMENT
 * -----------------------------------------------------------------------
 * A server can crash at any of these exact moments, and each one creates
 * a DIFFERENT kind of mess:
 *   - before it even receives a message
 *   - after receiving it, but before processing it
 *   - after processing it, but before sending back a response
 *   - after sending a response, but before the sender actually gets it
 *
 * So when a service comes back online, it needs to figure out: "wait,
 * where exactly did I stop? Did I already do this, or not?" Getting that
 * recovery logic right is a big part of what makes this hard.
 *
 * -----------------------------------------------------------------------
 * 2.4 THE CAP THEOREM (pick your poison during a network split)
 * -----------------------------------------------------------------------
 * You've probably heard "CAP theorem: pick two out of three." The more
 * USEFUL way to think about it: when the network is actually broken
 * (a "partition" — some nodes can't reach others), your system has to
 * choose between:
 *
 *   - Consistency:  everyone sees the same, correct, up-to-date answer
 *   - Availability: every request still gets SOME answer (even if a
 *                   little stale or possibly wrong)
 *
 *   (the third letter, Partition tolerance, isn't really a choice —
 *   real networks WILL partition sometimes, so you have to handle it)
 *
 * PICTURE (the trade-off triangle):
 *
 *        Consistency               Availability
 *              \                       /
 *               \      CA Systems     /
 *                \  (single-node DB) /
 *      CP Systems \________________ / AP Systems
 *   (Spanner, HBase)  Partition    (Cassandra, DynamoDB)
 *                     Tolerance
 *
 * For strong distributed transactions, the safe default is usually:
 * choose CONSISTENCY over availability. If the coordinator can't reach
 * enough participants, it should just BLOCK, REJECT, or TIME OUT —
 * rather than risk committing something that breaks atomicity.
 *
 *
 * ============================================================================
 * 3. THE BIG QUESTION FIRST: CAN YOU AVOID THIS ENTIRELY?
 * ============================================================================
 *
 * Before picking a fancy protocol, ask the SENIOR-ENGINEER question:
 * "Do I even NEED a distributed transaction here?"
 *
 * Try to keep strongly-related writes inside ONE database transaction if
 * at all possible. If the workflow truly HAS to cross service
 * boundaries, then pick the WEAKEST coordination mechanism that still
 * protects your business rule — don't reach for the heaviest tool by
 * default.
 *
 * The approaches roughly fall into 3 buckets:
 *
 *   BUCKET 1 — AVOID ENTIRELY
 *     - keep it in a single database
 *     - redesign service boundaries so it doesn't need to cross them
 *     - use idempotent operations
 *
 *   BUCKET 2 — EVENTUAL CONSISTENCY (accept a short delay, converge later)
 *     - Saga Pattern
 *     - Outbox Pattern
 *     - Event Sourcing
 *
 *   BUCKET 3 — ATOMIC COORDINATION (try to force "all or nothing" directly)
 *     - Two-Phase Commit (2PC)
 *     - Consensus-backed databases
 *     - Three-Phase Commit (3PC) — mostly theoretical
 *
 * Let's go through the main ones in detail.
 *
 *
 * ============================================================================
 * 4. TWO-PHASE COMMIT (2PC) — "everyone votes, then everyone acts"
 * ============================================================================
 *
 * This is the classic protocol. One node is the "coordinator," and it
 * runs the whole decision process in TWO phases.
 *
 * PHASE 1 — PREPARE (voting):
 *   The coordinator asks everyone: "can you commit this?"
 *   Each participant does the work LOCALLY, grabs any locks it needs,
 *   writes it to its log — but does NOT actually commit yet. Then it
 *   votes: YES ("I can definitely commit") or NO ("something's wrong").
 *
 * PHASE 2 — COMMIT (the decision):
 *   - If EVERYONE voted YES -> coordinator tells everyone: "COMMIT"
 *   - If ANYONE voted NO    -> coordinator tells everyone: "ABORT"
 *   Everyone carries out the decision and confirms back.
 *
 * PICTURE:
 *
 *   PHASE 1:
 *     [Coordinator] --PREPARE--> [DB A] --YES-->
 *                   --PREPARE--> [DB B] --YES-->
 *                   --PREPARE--> [DB C] --YES-->
 *
 *   PHASE 2 (all said yes):
 *     [Coordinator] --COMMIT--> [DB A] --ACK-->
 *                   --COMMIT--> [DB B] --ACK-->
 *                   --COMMIT--> [DB C] --ACK-->
 *
 *   Once a participant votes YES, it's making a PROMISE: "I will commit
 *   if told to, no matter what happens to me in the meantime" — it
 *   writes enough to its own log to guarantee that, even after a crash.
 *
 * THE PROBLEM: BLOCKING (this is the big weakness)
 *   After voting YES, a participant is STUCK holding its locks, waiting
 *   for the final word from the coordinator. If the coordinator crashes
 *   right after collecting all the YES votes — but BEFORE telling
 *   anyone the final decision — everyone is frozen:
 *
 *   PICTURE:
 *
 *      [Coordinator: CRASHED]
 *             |         |         |
 *      [DB A: PREPARED, locks HELD, waiting...]
 *      [DB B: PREPARED, locks HELD, waiting...]
 *      [DB C: PREPARED, locks HELD, waiting...]
 *
 *   They CAN'T commit (maybe the coordinator actually decided to abort).
 *   They CAN'T abort either (maybe the coordinator already told someone
 *   else to commit). So they just... wait. And this can cascade — other
 *   transactions that need those same locked rows start blocking too,
 *   which can freeze big chunks of your whole system.
 *
 * WHEN TO ACTUALLY USE 2PC:
 *   - within a single data center, where network splits are rare
 *   - short-lived transactions (small blocking window if things go wrong)
 *   - when your databases support "XA-style" distributed transactions
 *   - when strong consistency really matters more than availability
 *
 *
 * ============================================================================
 * 5. THREE-PHASE COMMIT (3PC) — "2PC's theoretical cousin"
 * ============================================================================
 *
 * 3PC tries to fix 2PC's blocking problem by adding an EXTRA phase in
 * the middle, giving participants a bit more info before committing.
 *
 * THE 3 PHASES:
 *   1. CanCommit  -> coordinator just checks "could you commit this?"
 *   2. PreCommit  -> if everyone said yes, coordinator tells them "get
 *                    ready" — this signals that EVERYONE already agreed
 *   3. DoCommit   -> coordinator finally says "go ahead, commit for real"
 *
 * PICTURE (comparing to 2PC):
 *
 *   2PC:  PREPARE -> COMMIT
 *   3PC:  CanCommit -> PreCommit -> DoCommit
 *
 * THE IDEA: a participant sitting in "PreCommit" already KNOWS that
 * everyone agreed — so, under certain timing assumptions, it can safely
 * move forward on its own after a timeout, instead of freezing forever
 * like in 2PC.
 *
 * THE CATCH: those "timing assumptions" (like messages always arriving
 * within a known time limit, and no network partitions) are TOO STRONG
 * for real-world systems. During an actual network split, DIFFERENT
 * groups of participants might time out and make DIFFERENT decisions —
 * some commit, some abort — leaving your data inconsistent anyway.
 *
 * WHEN TO USE IT: basically never in real production systems. It's
 * mostly a teaching tool to understand WHY 2PC blocks. In interviews,
 * mentioning 3PC as "the practical answer" is a red flag — real systems
 * reach for 2PC (in narrow cases), consensus databases, sagas, TCC, or
 * the outbox pattern instead.
 *
 *
 * ============================================================================
 * 6. SAGA PATTERN — "break it into small steps, undo if needed"
 * ============================================================================
 *
 * Instead of trying to force multiple systems to commit together all at
 * once, a Saga BREAKS the big transaction into a chain of small, LOCAL
 * transactions. Each step commits immediately for real, and then
 * publishes an event that kicks off the next step.
 *
 * If something fails partway through, the saga runs "compensating"
 * steps to UNDO the work that already happened.
 *
 * This trades "instant atomicity" for "high availability" — the system
 * is briefly inconsistent while the saga runs, but it should eventually
 * converge to a correct final state (as long as retries/compensations work).
 *
 * EXAMPLE — placing an order (4 steps):
 *
 *   [T1: Create Order] -> [T2: Reserve Inventory] -> [T3: Process Payment] -> [T4: Ship Order]
 *
 * What if Payment (T3) FAILS after T1 and T2 already succeeded? The saga
 * has to walk BACKWARD and undo what it already did:
 *
 *   FORWARD:      T1 (success) -> T2 (success) -> T3 (FAILED)
 *   COMPENSATION: <- C2: Release Inventory <- C1: Cancel Order
 *
 * IMPORTANT REALITY CHECK: "undoing" isn't always clean. You can't
 * "unsend" an email. Refunding a payment is NOT the same as never
 * charging someone in the first place (fees, timing, trust — all
 * different). Designing GOOD compensation logic takes real thought.
 *
 * TWO WAYS TO COORDINATE A SAGA:
 *
 *   A) ORCHESTRATION — one central "boss" tells everyone what to do
 *
 *      PICTURE:
 *        [Saga Orchestrator] --1. Create--> [Order Service] --Done-->
 *                            --2. Reserve--> [Inventory Service] --Done-->
 *                            --3. Charge--> [Payment Service] --Done-->
 *
 *   B) CHOREOGRAPHY — no central boss; each service just reacts to events
 *
 *      PICTURE:
 *        [Order Service] --"OrderCreated" event--> [Inventory Service]
 *        [Inventory Service] --"InventoryReserved" event--> [Payment Service]
 *        [Payment Service] --"PaymentProcessed" event--> (next step)
 *
 *   TRADE-OFFS:
 *
 *     Aspect        | Orchestration                        | Choreography
 *     ---------------|-----------------------------------------|-----------------------------------
 *     Complexity      | Centralized — easier to follow/debug     | Spread out — harder to trace end-to-end
 *     Coupling        | The "boss" knows about every service      | Services only care about events they subscribe to
 *     Failure risk    | The orchestrator itself must be reliable  | No single coordinator, but shared topics/schemas still matter
 *     Adding a step   | Just update the orchestrator               | Add a new service that listens for existing events
 *
 *   RULE OF THUMB: Orchestration is better for complex workflows where
 *   you want clear visibility. Choreography is better when you want
 *   loose coupling and expect the workflow to keep growing over time.
 *
 * WHEN TO USE SAGAS:
 *   - microservices where you don't strictly need cross-service atomicity
 *   - long-running workflows (could take minutes or even hours)
 *   - when you CAN define a real, meaningful "undo" for every step
 *   - when availability matters more than a brief window of inconsistency
 *
 *
 * ============================================================================
 * 7. TCC PATTERN (TRY-CONFIRM-CANCEL) — "reserve first, decide later"
 * ============================================================================
 *
 * TCC is like a Saga's more cautious cousin. Instead of committing each
 * step immediately (like a saga does), it first RESERVES resources, and
 * only actually finalizes everything once ALL reservations succeeded.
 *
 * EVERY operation has 3 phases:
 *   - Try:     check that it's possible, and RESERVE the resource
 *              (but don't commit for real yet)
 *   - Confirm: actually finalize the reservation (must be safe to retry!)
 *   - Cancel:  release the reservation if anything went wrong
 *
 * PICTURE:
 *
 *   TRY PHASE (reserve everywhere):
 *     [Order: Reserve order ID] [Inventory: Reserve 5 items] [Payment: Reserve $100]
 *
 *   IF ALL TRY SUCCEED -> CONFIRM PHASE:
 *     [Order: Create order] [Inventory: Deduct 5 items] [Payment: Charge $100]
 *
 *   IF ANY TRY FAILS -> CANCEL PHASE:
 *     [Order: Release order ID] [Inventory: Release 5 items] [Payment: Release $100]
 *
 * KEY IDEA: the "Try" phase should do all the HEAVY validation and
 * reserve everything up front, so "Confirm" is just a tiny, safe,
 * retryable step at the end (since it can still time out or fail, and
 * needs to be safe to run more than once).
 *
 * TCC vs SAGA — what's actually different:
 *
 *   Aspect        | TCC                                    | Saga
 *   ---------------|-------------------------------------------|-------------------------------------
 *   Reservation     | Explicit "Try" phase holds the resource    | Each step commits right away, for real
 *   Isolation        | Better — stuff sits in "pending" state    | Worse — half-finished results may be visible to others
 *   Complexity       | Higher — 3 operations per service          | Lower — just 2 operations per service
 *   Rollback         | "Cancel" cleanly releases the reservation | Compensations have to UNDO already-committed work
 *
 * The real difference is ISOLATION: in a Saga, once "Create Order" runs,
 * that order might already be VISIBLE to other parts of the system. In
 * TCC, it just sits there in a "reserved/pending" state until Confirm runs.
 *
 * WHEN TO USE TCC:
 *   - financial transactions (reserve funds before actually committing)
 *   - anywhere half-finished/visible intermediate states would be a problem
 *   - short-lived transactions where holding a reservation briefly is fine
 *
 *
 * ============================================================================
 * 8. OUTBOX PATTERN — "write the data AND the event together, safely"
 * ============================================================================
 *
 * This pattern solves a DIFFERENT but very common problem: reliably
 * updating a database AND publishing an event about it, without needing
 * any fancy cross-system coordination.
 *
 * THE PROBLEM IT SOLVES ("the dual-write problem"):
 *   Normally you'd do TWO separate things: (1) write to the database,
 *   then (2) publish an event to a message broker. But these are two
 *   INDEPENDENT operations that can fail SEPARATELY:
 *
 *   PICTURE:
 *
 *     [Service] --1. Write--> [Database] SUCCESS ✓
 *               --2. Publish--> [Message Broker] FAILED ✗
 *
 *   If the database write succeeds but publishing the event fails, your
 *   database is updated, but NOBODY else finds out about it. Now you're
 *   quietly inconsistent. (And swapping the order — publish first, THEN
 *   write — has the opposite problem: you'd tell everyone about
 *   something that then never actually happened.)
 *
 * HOW THE OUTBOX PATTERN FIXES THIS:
 *   Instead of writing to the database AND separately publishing to a
 *   broker, you write BOTH the actual business data AND a row
 *   describing the event, into the SAME database, in ONE local
 *   transaction. A separate background process then reads that
 *   "outbox" table and publishes events to the broker on its own time.
 *
 *   PICTURE:
 *
 *      [Order Service] --(single local transaction)--> [Orders Table]
 *                                                    -> [Outbox Table]
 *
 *      [Outbox Relay: background process] --reads from--> [Outbox Table]
 *                                        --publishes to--> [Message Broker]
 *
 *   STEP BY STEP:
 *     1. In ONE local transaction, write the business data AND insert a
 *        row into the outbox table describing the event
 *     2. A separate background process reads unpublished rows from the
 *        outbox table and sends them to the message broker
 *     3. Once successfully published, mark that row as "processed"
 *
 *   Since the outbox row and the business data are written in the SAME
 *   local transaction, they're either BOTH saved or NEITHER is — no
 *   coordination between different systems needed. If the relay process
 *   or broker is briefly down, it just retries later from the durable
 *   outbox table — nothing gets silently lost.
 *
 *   ONE SIDE EFFECT: the same event might get delivered MORE than once,
 *   so anything CONSUMING these events needs to be idempotent (safe to
 *   process the same event twice without messing things up).
 *
 * WHEN TO USE THE OUTBOX PATTERN:
 *   - event-driven systems where services talk to each other via events
 *   - anywhere reliable event publishing really matters
 *   - microservices needing eventual consistency
 *   - as the underlying foundation for building Sagas via choreography
 *
 *
 * ============================================================================
 * 9. COMPARING ALL THE APPROACHES
 * ============================================================================
 *
 *   Approach | Outcome Model            | Availability | Latency | Complexity | Best For
 *   ----------|-----------------------------|-----------------|-----------|--------------|--------------------------------
 *   2PC       | Atomic commit               | Low             | Medium    | Medium       | Same data center, XA-compatible systems
 *   3PC       | Unsafe during partitions    | Medium          | High      | High         | Mostly theoretical / academic
 *   Saga      | Eventual / compensating     | High            | Varies    | High         | Microservices, long-running workflows
 *   TCC       | Reservation-based           | Medium          | Medium    | High         | Financial/inventory systems needing holds
 *   Outbox    | Eventual                    | High            | Low       | Low          | Event-driven architectures
 *
 *   REMEMBER: TCC is NOT magic atomic commit. It only works when Try,
 *   Confirm, Cancel, retries, AND expiry are ALL carefully designed.
 *
 * DECISION CHECKLIST (walk through in order):
 *
 *   1. Do you truly need STRONG consistency (all-or-nothing, right now)?
 *      -> No, eventual consistency is fine -> keep reading below
 *      -> Yes -> can everything fit in a SINGLE database? -> use local
 *         transactions (simplest and best option if it's possible!)
 *      -> Yes, but it MUST cross services -> do all participants support
 *         XA-style coordination? -> use 2PC. If not, and you can define
 *         clean reservations -> use TCC.
 *
 *   2. If eventual consistency is fine: do you need reliable EVENT
 *      publishing tied to a database write? -> use the Outbox Pattern.
 *
 *   3. If you need a whole multi-step DISTRIBUTED WORKFLOW with eventual
 *      consistency? -> use a Saga.
 *
 *   THE #1 QUESTION, ALWAYS: can you avoid a distributed transaction
 *   entirely by restructuring your data or service boundaries? If yes,
 *   do that first — local transactions are simpler, faster, and more
 *   reliable than ANY distributed alternative.
 *
 *
 * ============================================================================
 * 10. BEST PRACTICES (things every real implementation needs)
 * ============================================================================
 *
 * -----------------------------------------------------------------------
 * 10.1 PREFER LOCAL TRANSACTIONS — always ask first
 * -----------------------------------------------------------------------
 * Before reaching for ANY distributed transaction pattern, ask:
 *   - Can the data that needs to change together just live TOGETHER
 *     (same database)?
 *   - Can you redraw your service boundaries so this workflow doesn't
 *     cross them?
 *   - Could ONE database with multiple schemas work, instead of multiple
 *     separate databases?
 * A LOT of systems that seem to "need" distributed transactions can
 * actually be redesigned to avoid them. Not always possible — but always
 * worth checking first.
 *
 * -----------------------------------------------------------------------
 * 10.2 DESIGN FOR IDEMPOTENCY — retries WILL happen
 * -----------------------------------------------------------------------
 * Networks fail, timeouts happen, and sometimes you genuinely don't know
 * if an operation succeeded. So: any operation that MIGHT get retried
 * should be "idempotent" — running it twice should have the SAME effect
 * as running it once.
 *
 *   // BAD — calling this twice adds 200 total (unsafe!)
 *   function incrementBalance(userId, amount) {
 *     user.balance += amount;
 *   }
 *
 *   // GOOD — calling this twice has the same effect as calling it once
 *   function updateBalance(userId, requestId, newBalance) {
 *     if (!alreadyProcessed(requestId)) {
 *       user.balance = newBalance;
 *       markProcessed(requestId);
 *     }
 *   }
 *
 * The trick: attach a UNIQUE request ID to every operation, and keep
 * track of which request IDs you've already handled.
 *
 * -----------------------------------------------------------------------
 * 10.3 ALWAYS USE TIMEOUTS — never wait forever
 * -----------------------------------------------------------------------
 * Every call to another service needs a timeout. Without one, a single
 * slow or dead service can freeze your ENTIRE system.
 *
 *   // DANGEROUS — could hang forever
 *   const result = remoteService.call();
 *
 *   // SAFE — bounded wait
 *   try {
 *     const result = remoteService.call({ timeout: 5000 }); // ms
 *   } catch (e) {
 *     handleFailure();
 *   }
 *
 * Getting the timeout VALUE right matters: too short and you'll fail
 * perfectly healthy requests; too long and failures start cascading
 * outward. Start with a few seconds and adjust based on real behavior.
 *
 * -----------------------------------------------------------------------
 * 10.4 BUILD COMPENSATING TRANSACTIONS CAREFULLY
 * -----------------------------------------------------------------------
 * Example: a saga creates an order AND sends a confirmation email. If
 * payment fails LATER, you can cancel the order — but the email is
 * already sent. You can't take that back.
 *
 * SOLUTIONS:
 *   - delay IRREVERSIBLE actions (like sending emails) until you're
 *     confident the whole saga will succeed
 *   - use "pending"/"reserved" states instead of committing immediately
 *   - hold notifications until the very end of the saga
 *   - for anything TRULY impossible to undo, ask if it even belongs
 *     inside the saga at all
 *   - also plan for what happens if the COMPENSATION itself fails — you
 *     need monitoring and a manual-intervention plan for that case too
 *
 * -----------------------------------------------------------------------
 * 10.5 MONITOR AND ALERT
 * -----------------------------------------------------------------------
 * Distributed transactions fail in ways local ones simply don't. Track:
 *   - success/failure rate, per transaction type
 *   - latency percentiles (p50, p95, p99)
 *   - how often compensations get triggered
 *   - transactions STUCK in an in-between state
 * Set up alerts specifically for "stuck" transactions — something that's
 * been "in progress" for hours is a clear sign something's broken and
 * needs a human to look at it.
 *
 * -----------------------------------------------------------------------
 * 10.6 TEST FAILURE SCENARIOS ON PURPOSE
 * -----------------------------------------------------------------------
 * Don't just hope your saga's compensation logic works — TEST it:
 *   - kill a service mid-transaction, verify compensation runs correctly
 *   - inject network delays, verify your timeouts actually trigger
 *   - simulate a network partition, verify you don't end up inconsistent
 * These tests often reveal subtle bugs that normal, happy-path testing
 * never catches.
 *
 *
 * ============================================================================
 * 11. COMMON PITFALLS (mistakes people make again and again)
 * ============================================================================
 *
 * -----------------------------------------------------------------------
 * PITFALL 1: IGNORING PARTIAL FAILURES
 * -----------------------------------------------------------------------
 * In a distributed system, SOME steps might succeed while OTHERS fail.
 * Code that assumes "either everything worked or nothing did" is
 * dangerous:
 *
 *   // DANGEROUS — no idea what actually succeeded after the error
 *   try {
 *     serviceA.update();
 *     serviceB.update();
 *     serviceC.update();
 *   } catch (e) {
 *     // Did A succeed? Did B? We genuinely don't know!
 *     log.error("Something failed");
 *   }
 *
 *   // SAFE — track what actually completed, only undo THAT
 *   const completed = [];
 *   try {
 *     serviceA.update(); completed.push("A");
 *     serviceB.update(); completed.push("B");
 *     serviceC.update(); completed.push("C");
 *   } catch (e) {
 *     compensate(completed); // only undo what we KNOW actually happened
 *   }
 *
 * ALWAYS ask yourself: "if this line throws right now, what STATE is my
 * system actually left in?"
 *
 * -----------------------------------------------------------------------
 * PITFALL 2: ASSUMING NETWORKS ARE RELIABLE
 * -----------------------------------------------------------------------
 * Messages can be lost, delayed, duplicated, or reordered. To stay
 * correct: use acknowledgments or durable offsets to confirm things were
 * actually received/processed, add retries with exponential backoff for
 * transient failures, and make EVERY operation idempotent so duplicate
 * messages don't cause double-processing.
 *
 * Ironic but true: a network that's "mostly reliable" is actually WORSE
 * than one that's completely unreliable — because the bugs only show up
 * occasionally, making them brutally hard to reproduce and debug.
 *
 * -----------------------------------------------------------------------
 * PITFALL 3: HOLDING LOCKS TOO LONG
 * -----------------------------------------------------------------------
 * Every second you hold a lock, other operations are stuck waiting. It
 * gets especially bad if you hold a lock WHILE waiting on a slow,
 * external call:
 *
 *   // DANGEROUS — lock is held during a slow external call
 *   lock.acquire();
 *   const data = externalService.slowCall(); // takes 5 seconds!
 *   database.update(data);
 *   lock.release();
 *
 *   // BETTER — do the slow part FIRST, lock only for the fast part
 *   const data = externalService.slowCall(); // no lock held here
 *   lock.acquire();
 *   database.update(data); // milliseconds
 *   lock.release();
 *
 * RULE: do all your SLOW work before grabbing a lock, and release the
 * lock as soon as humanly possible.
 *
 * -----------------------------------------------------------------------
 * PITFALL 4: NOT PLANNING FOR ROLLBACK FROM DAY ONE
 * -----------------------------------------------------------------------
 * Compensating transactions are MUCH harder to bolt on later. While
 * you're building the "happy path" of a saga, writing the undo-logic
 * feels like extra work you can skip for now — but skip it, and you'll
 * end up with operations that simply CANNOT be undone later.
 *
 * The RIGHT time to design the "undo" is the SAME time you design the
 * original action. Always ask: "if this step succeeds but the overall
 * transaction later fails, how exactly do we undo it?" If you don't have
 * a good answer, that's a sign to rethink the design right now — not later.
 *
 *
 * ============================================================================
 * BIG TAKEAWAYS (remember these if nothing else)
 * ============================================================================
 *
 *   1. In ONE database, "all or nothing" is free (ACID). Across
 *      services, it's something YOU have to design and build.
 *
 *   2. The hard part isn't any single failure — it's that networks lose
 *      messages, services crash at random moments, and nobody can ever
 *      be 100% sure what the OTHER side actually did (the "Two Generals"
 *      problem).
 *
 *   3. ALWAYS ask first: "can I avoid needing a distributed transaction
 *      at all?" Restructuring boundaries or keeping related data
 *      together is usually simpler and more reliable than any protocol.
 *
 *   4. 2PC gives strong atomicity but can BLOCK badly if the coordinator
 *      crashes at the wrong moment — use it only in narrow, low-risk cases.
 *
 *   5. 3PC is mostly a teaching tool — it doesn't survive real network
 *      partitions, so don't propose it as a real-world solution.
 *
 *   6. Sagas break a big transaction into small local steps with
 *      compensations — great for microservices, but "undo" logic needs
 *      real thought (you can't unsend an email).
 *
 *   7. TCC adds an explicit "reserve first" phase for better isolation
 *      — great for money/inventory, but adds real complexity (3 phases
 *      instead of 2).
 *
 *   8. Outbox pattern solves the "did my database write AND my event
 *      publish both succeed?" problem by writing both in ONE local
 *      transaction, then relaying events in the background.
 *
 *   9. No matter which pattern you pick: make everything idempotent, use
 *      timeouts everywhere, track partial failures explicitly, monitor
 *      stuck transactions, and TEST your failure paths on purpose.
 *
 *  10. Design your "undo" logic at the SAME time as your "do" logic —
 *      not as an afterthought once things are already broken.
 *
 * ============================================================================
 */